# Interactive Prototype

Goal: Create a functional MVP for field route planning and management.

Stages:
1. Region Input & Sub-Division
2. Automatic Target Search & Routing
3. Manual Adjustment
4. Select & Execute Plans

## Region Input & Sub-Division

- Get region outline
- Divide into work cells
- Tentative plan for depot locations
- 

### Dummy Region Shapefile

At this point, we do not actually have a shapefile of the target region. The following two Jupyter cells will generate one using the outline of the orthophoto we have created. 

Load this as the "shapefile" which will define our working region. 

In [80]:
region_image_path = '../input/IGNORE_Brewster-2024-all-orthophoto-UTM-32613.tif'
region_contour_shapefile = '../input/interactive_proto/region_contour.shp'
region_contour_geojson = '../input/interactive_proto/region_contour.geojson'


region_crs = 32613 # Use this everywhere for consistency
visualization_crs = 4326 # Use this when we need leaflet visualizations
simplification_tolerance = 5


In [119]:
import sys
import geopandas as gpd
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from plant_search.load_image import load_image
from plant_search.region_partition import extract_region_contour, simplify_polygon

# image, transform, bounds, image_crs = load_image(region_image_path)

# Get region contour, emulating shapefile input
region_contour_gpd = extract_region_contour(region_image_path)
contour_gdf_projected = region_contour_gpd.to_crs(epsg=region_crs)  # Example: UTM Zone 14N
region_contour = simplify_polygon(contour_gdf_projected.geometry.iloc[0], simplification_tolerance)
region_contour_gdf = gpd.GeoDataFrame({"geometry": [region_contour]}, crs=region_crs) # Convert to GDF
# region_contour_gdf = gpd.GeoDataFrame({"geometry": [region_contour]}) # Convert to GDF

# Save as shape file
region_contour_gdf.to_file(region_contour_shapefile, driver="ESRI Shapefile")

# Prove it can be loaded
loaded_gdf = gpd.read_file(region_contour_shapefile)
print(f"Loaded shapefile CRS: {loaded_gdf.crs}")
print(type(loaded_gdf))

# Convert to GeoJSON
loaded_gdf.to_crs(epsg=4326, inplace=True)
loaded_gdf.to_file(region_contour_geojson, driver="GeoJSON")
print(f"Written geoJSON CRS: {loaded_gdf.crs}")
print(type(loaded_gdf))

Loaded shapefile CRS: EPSG:32613
<class 'geopandas.geodataframe.GeoDataFrame'>
Written geoJSON CRS: EPSG:4326
<class 'geopandas.geodataframe.GeoDataFrame'>


### Create Voronoi Partitioning, Solve for Depots

In [154]:
target_area_acres = 0.5
# target_area_acres = 1.5
# target_area_acres = 2.5

target_area_sqm = target_area_acres * 4046.86
max_iterations = 15

voronoi_partition_filename = '../input/interactive_proto/voronoi_partition.geojson'
voronoi_centroids_filename = '../input/interactive_proto/voronoi_centroids.geojson'

In [ ]:
from plant_search.region_partition import centroidal_voronoi_tessellation

region_outline_gdf = gpd.read_file(region_contour_shapefile)
simplified_polygon = region_outline_gdf.geometry.iloc[0]
# print(loaded_gdf.crs)

num_points = int(simplified_polygon.area / target_area_sqm) # How many cells to generate

cell_gdf = centroidal_voronoi_tessellation(simplified_polygon, num_points, max_iterations)

cell_gdf_4326 = cell_gdf.copy().to_crs(visualization_crs)
# print(cell_gdf_4326.crs)

# Create a copy with only the 'geometry' column (Voronoi polygons)
voronoi_gdf = cell_gdf_4326.copy().drop(columns=["cell_centroid"])
voronoi_gdf.to_crs(visualization_crs, inplace=True)
voronoi_gdf.to_file(voronoi_partition_filename, driver="GeoJSON")

# Create a copy with only the 'cell_centroid' column and set it as the active geometry
centroid_gdf = cell_gdf_4326.copy().drop(columns=["geometry"])
centroid_gdf.set_geometry("cell_centroid", inplace=True)

centroid_gdf.set_crs(region_crs, inplace=True)  # Reset the CRS explicitly
centroid_gdf.to_crs(visualization_crs, inplace=True)  # Reset the CRS explicitly
centroid_gdf.to_file(voronoi_centroids_filename, driver="GeoJSON")

Reached maximum iterations without full convergence.
EPSG:4326


In [ ]:
import geopandas as gpd
from ipyleaflet import Map, GeoJSON, LayersControl
from shapely.geometry import mapping, shape
import json

# Load the GeoJSON region outline
with open(region_contour_geojson, "r") as f:
    region_contour_data = json.load(f)
region_geometry = shape(region_contour_data['features'][0]['geometry'])
region_center = region_geometry.centroid

# Load Voronoi polygons
with open(voronoi_partition_filename, "r") as f:
    voronoi_data = json.load(f)

# Load centroids
with open(voronoi_centroids_filename, "r") as f:
    centroid_data = json.load(f)

# print(region_contour_data)
# print(voronoi_data)
# print(centroid_data)


m = Map(center=(region_center.y, region_center.x), zoom=16)

# Add the region border to the map
region_layer = GeoJSON(
    data=region_contour_data, 
    style={'color': 'green', 'fillOpacity': 0.2, 'weight': 3},
    name=region_contour_data['name'])
m.add_layer(region_layer)

# Add Voronoi polygons
voronoi_layer = GeoJSON(
    data=voronoi_data, 
    style={'color': 'blue', 'fillColor': 'lightblue', 'opacity': 0.5, 'weight': 2},
    name=voronoi_data['name'])
m.add_layer(voronoi_layer)

# Add centroids
centroid_layer = GeoJSON(
    data=centroid_data, 
    style={'color': 'red', 'radius': 1, 'fillOpacity': 1.0},
    name=centroid_data['name'])
centroid_layer
centroid_layer.visible = False  # Set layer to hidden
m.add_layer(centroid_layer)

m.add_control(LayersControl()) # Add layer control
m # Display the map

Map(center=[30.248931657196884, -103.60192091362077], controls=(ZoomControl(options=['position', 'zoom_in_text…